In [1]:
from dd_solvers import *
from experiments import *
import torch
import pandas as pd

AMGX version 2.5.0
Built on Oct 12 2026, 08:12:03
Compiled with CUDA Runtime 12.6, using CUDA driver 13.0
The AMGX_initialize_plugins API call is deprecated and can be safely removed.


/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
fine_m = 7
mesh_family = UniformMeshes(d=2, m=fine_m)

In [3]:
common_kwargs = {
    "test_case": continuous_coefficient_2d,
    "polynomial_degree": 1,
}

preconditioner_factory_kwargs = {
    **common_kwargs,
    "setup_repetitions": 1,
    "solution_warmup_steps": 10,
    "solution_measurement_steps": 10,
    "solution_repetitions": 10,
}

In [4]:
preconditioner_factory = ExperimentFactory(
    **preconditioner_factory_kwargs, mesh_family=mesh_family
)

In [5]:
ms = [
    (f"C{solvers_m}", f"C{solvers_m}", f"S{fine_m}")
    for solvers_m in range(2, fine_m + 1)
] + [
    (f"S{solvers_m}", f"S{solvers_m}", f"S{fine_m}")
    for solvers_m in range(2, fine_m + 1)
]

In [6]:
preconditioners = [
    AdditiveSchwarz(prec, BatchCUDSS(), CUDSS(), collect_timings=True)
    for prec in [torch.float32, torch.float64]
]

In [7]:
for coarse_m, solvers_m, fine_m_str in ms:
    for preconditioner in preconditioners:
        preconditioner_factory.add(
            Experiment(
                coarse_m=coarse_m,
                solvers_m=solvers_m,
                fine_m=fine_m_str,
                solver=preconditioner,
            )
        )

In [8]:
df = preconditioner_factory.run()

Running experiments:   0%|          | 0/24 [00:00<?, ?it/s]/workspace/src/dd_solvers/dd_solvers/solvers.py:117: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  return torch.sparse_csr_tensor(
/workspace/src/experiments/experiments/experiments.py:206: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  fine_to_dofs = torch.as_tensor(
Running experiments: 100%|██████████| 24/24 [0

In [18]:
df.to_csv("../results/benchmark_local_cudss_solver.csv")